# 49. Group Anagrams
**Difficulty:** 🟡 Medium · **Topic:** String · **LeetCode:** https://leetcode.com/problems/group-anagrams/

## 💡 Concepts

**Core concept(s):** A **hash map** whose key is a *canonical form* shared by all anagrams.

**Why it applies here:** Anagrams share the same letters, so if we turn each word into a form that ignores order — its sorted letters, or its letter counts — all anagrams get the **same key**. Group words by that key in a hash map.

**Key intuition:** Give every word a fingerprint that anagrams share, then bucket words by fingerprint.

---

### 📚 What is a Hash Map / Hash Set?
A **hash map** (Python `dict`) stores **key → value** pairs; a **hash set** (`set`) stores unique keys. Both use a *hash function* to jump straight to a slot instead of scanning.
- **Operations & complexity:** insert / lookup / delete are **O(1) on average**.
- **In Python:** `dict` for counts/mappings, `set` for "have I seen this?". `collections.Counter` counts items for you.

### 📚 What is Sorting (as a tool)?
**Sorting** reorders characters so structure becomes obvious (e.g. two anagrams become the *same* string once sorted).
- **Complexity:** Python's `sorted()` is **O(k log k)** for length k.

---

**Prerequisite knowledge:**
- Hash map of key -> list.
- Making a canonical key (sorted string or count tuple).

## 📝 Problem

Group the words that are anagrams of each other.

**Example**
```
["eat","tea","tan","ate","nat","bat"]
-> [["eat","tea","ate"], ["tan","nat"], ["bat"]]  (any order)
```

### Approach 1 — Sorted-String Key (worst)

**Idea:** Use each word's **sorted letters** as the key; anagrams sort to the same string.

**Time complexity:** `O(n · k log k)` for n words of length k.

**Space complexity:** `O(n · k)`.

In [ ]:
from collections import defaultdict
from typing import List

def group_sort(strs: List[str]) -> List[List[str]]:
    groups = defaultdict(list)             # fingerprint -> list of words with that fingerprint
    for w in strs:
        key = "".join(sorted(w))           # sorted letters: all anagrams share this key
        groups[key].append(w)              # drop the word into its bucket
    return list(groups.values())           # each bucket is one anagram group

### Approach 2 — Letter-Count Key (optimal)

**Idea:** Use a tuple of 26 letter counts as the key — no sorting needed.

**Time complexity:** `O(n · k)`.

**Space complexity:** `O(n · k)`.

In [ ]:
from collections import defaultdict
from typing import List

def group_count(strs: List[str]) -> List[List[str]]:
    groups = defaultdict(list)             # fingerprint -> list of words
    for w in strs:
        counts = [0] * 26                  # how many of each letter a..z this word has
        for c in w:
            counts[ord(c) - ord("a")] += 1 # tally the letter (a=0, b=1, ...)
        groups[tuple(counts)].append(w)    # a tuple is hashable, so it works as a dict key
    return list(groups.values())

In [ ]:
# Correctness check (compare as a set of frozensets, since order is free)
def norm(groups):
    return {frozenset(g) if len(set(g)) == len(g) else tuple(sorted(g)) for g in groups}

tests = [
    (["eat","tea","tan","ate","nat","bat"], 3),
    ([""], 1),
    (["a"], 1),
]
for strs, ngroups in tests:
    g1, g2 = group_sort(strs), group_count(strs)
    # same grouping regardless of approach
    key = lambda gs: sorted(sorted(g) for g in gs)
    print(f"{strs} -> {len(g1)} groups")
    assert key(g1) == key(g2) and len(g1) == ngroups, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit). Sub-millisecond rows are noisy — look at the trend.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    strs = ["dcba"] * n                     # all anagrams -> one big bucket, full work
    return (strs,)

solutions = {
    "sorted-key O(n*k log k)": group_sort,
    "count-key  O(n*k)      ": group_count,
}
sizes = [2000, 4000, 8000, 16000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Canonical key + hash map to group:** turn each item into a form its "siblings" share, then bucket by that form.
- **Counting beats sorting:** a count fingerprint avoids the log factor.
- **Signal:** "group / bucket things that are equivalent under some transformation".
- **Related problems:** Valid Anagram, Find All Anagrams, Group Shifted Strings.
- **Common pitfalls:** (1) using a `list` as a dict key (not hashable — use a `tuple`); (2) assuming input is lowercase-only.